# CNN 1D com fusão SEEDS: a otimização do trabalho de Souza (2023)

O código está no pacote `src/hsi/`, um arquivo por etapa; o mapa fica em `src/hsi/__init__.py`. Este notebook só importa o pacote, escolhe a configuração e roda.

Para preparar o ambiente:

```bash
uv venv --python 3.12
uv pip install -r requirements.txt
uv pip install -e . --no-deps
```

Os arquivos `.mat` das cenas ficam em `data/dataHSI/`.

In [ ]:
# Edições em src/hsi/ passam a valer sem reiniciar o kernel.
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt

from hsi.ambiente import check_versions, configure_determinism, select_device
from hsi.config import ExperimentConfig
from hsi.pipeline import run_experiment

check_versions()
device = select_device()
configure_determinism()

## 1. Uma execução

`ExperimentConfig` tem como padrão a configuração do notebook original: lote 32, taxa de aprendizado 0,001, dropout 0,5 e até 60 épocas, com SEEDS. O treino para depois de 10 épocas sem a acurácia de validação melhorar 0,001. Mude a cena ou qualquer campo na criação, por exemplo `ExperimentConfig(dataset="PU", segmenter="slic", train_fraction=0.10)`.

`run_experiment` prepara os dados, treina e devolve um dicionário com o modelo, o histórico e as métricas de teste (OA, AA e Kappa).

In [ ]:
config = ExperimentConfig(dataset="IP")
resultado = run_experiment(config, verbose=True)

In [ ]:
historico = resultado["historico"]
fig, (perda, acuracia) = plt.subplots(1, 2, figsize=(10, 3.5))
perda.plot(historico["loss"], label="treino")
perda.plot(historico["val_loss"], label="validação")
perda.set(xlabel="época", ylabel="perda")
acuracia.plot(historico["accuracy"], label="treino")
acuracia.plot(historico["val_accuracy"], label="validação")
acuracia.set(xlabel="época", ylabel="acurácia")
acuracia.legend()
plt.show()

## 2. Busca de hiperparâmetros

`automated_hpo` percorre o espaço de `hsi.hpo.HPO_SPACE` (taxa de aprendizado, dropout, weight decay, lote, componentes do PCA, segmentador e a configuração do segmentador) com uma das estratégias: `"random"`, `"tpe"` ou `"genetic"`. Cada trial é escolhido pela melhor acurácia de validação; o teste só é relatado.

In [ ]:
BUSCAR = False

if BUSCAR:
    from hsi.hpo import automated_hpo

    base = ExperimentConfig(dataset="IP", train_fraction=0.10, epochs=200, target=0.95)
    trials = automated_hpo(base, n_trials=8, search_epochs=10, strategy="tpe")
    display(trials.drop(columns="config"))

## 3. Paralelismo

A partição divide a cena em N faixas, treinadas ao mesmo tempo (`modo="independente"`) ou inferidas por um modelo só (`modo="replicado"`). A fusão concorrente é o esquema do orientador: uma thread segmenta enquanto as outras passam os pixels pelas convoluções e esperam o mapa só para a concatenação.

In [ ]:
PARALELISMO = False

if PARALELISMO:
    from hsi.particao import run_experiment_particionado

    particionado = run_experiment_particionado(
        ExperimentConfig(dataset="IP", epochs=25, batch_size=64),
        n=4, modo="independente", executor="process")
    print(f"OA {particionado['OA']:.4f} | bloco paralelo {particionado['tempo_paralelo']:.1f}s")

## 4. Experimentos da monografia

| Exp. | O que mede |
| --- | --- |
| 05 | segmentadores com a CNN fixa (contraexemplo) |
| 06 | cada segmentador com a CNN ajustada para ele |
| 08 | busca aleatória, TPE e algoritmo genético |
| 10 | configuração otimizada contra a original |
| 11 | partição da cena |
| 13 | pool frio contra pool quente |
| 14 | fusão concorrente |

Os resultados vão para `experimentos/resultados/`. Uma execução interrompida continua de onde parou, porque o experimento relê o CSV e pula o que já foi medido. Execuções longas rodam melhor no terminal:

```bash
.venv/bin/python src/executar.py 14 IP
```

In [ ]:
EXECUTAR = None          # por exemplo "14"
CENAS = ("IP",)

if EXECUTAR is not None:
    from hsi.experimentos import EXPERIMENTOS

    EXPERIMENTOS[EXECUTAR](*CENAS)

## 5. Figuras

As figuras do capítulo de resultados saem dos CSVs e vão para `tcc/TCC III - EDUARDO SAVIAN/src/assets/resultados/`.

In [ ]:
GERAR_FIGURAS = False

if GERAR_FIGURAS:
    from hsi.figuras import gerar

    gerar()